In [ ]:
from src.v4.torchdata import print_formatted_table, load_vals, unload_vals
from src.v5.problem import symbolic, log
from src.v6.problem import FunctionalSet, ConstraintSet, get_sharedvars
from graph.matrixview import render_incidence
from graph.graphutils import default_tree
import numpy as np

In [ ]:
d, A_solar, D_f = symbolic('d', 'A_{solar}', 'D_f')
alpha = 0.05
Geometry = ConstraintSet(
    D_f == (4*A_solar/(np.pi*(1-alpha)))**0.5,
    d == alpha * D_f,
    #A_solar == (1 - alpha) * np.pi * (D_f / 2)**2,
    idvals = [0,1]
)  

t_d, t_s, t_f, D_f, D_s, D_d, h_f, F_B, F_W, V_d = symbolic('t_d', 't_s', 't_f', 'D_f', 'D_s', 'D_d', 'h_f', 'F_B', 'F_W', 'V_d')
g, rho_w = 9.8, 1000  # m/s^2, kg/m^3
Hydro = ConstraintSet(
    V_d == np.pi / 4 * (t_d * D_d**2 +  t_s * D_s**2 + h_f * D_f**2),
    F_B == rho_w * V_d * g / 1000,
    F_W == F_B,
    idvals = [2,3,4]
) 

m_platform, m_solar, m_struct, m_batt, t_d = symbolic('m_{platform}', 'm_{solar}', 'm_{struct}', 'm_{batt}', 't_d')
m_comms, m_prop, rho, rho_h, eta_solar = 50.0, 50.0, 700, 2700, 10 
Mass = ConstraintSet(
    m_platform == F_W * 1000 / g,
    m_solar == eta_solar * A_solar,
    m_struct == m_platform - m_batt - m_solar - m_comms - m_prop,
    t_d == (4 / np.pi * m_struct - D_f**2 * t_f * rho - D_s**2 * t_s * rho) / (D_d**2 * rho_h),
    idvals = [5,6,7,8]
)

C_d, eta_m = 1.0, 0.75        # drag coeff (–), motor efficiency (–),
v, S_wd, S_ws, S_wf, S_w, P_move  = symbolic('v', 'S_{wd}', 'S_{ws}', 'S_{wf}', 'S_{w}', 'P_{move}')                               
Propulsion = ConstraintSet(
    S_wd  == np.pi*((D_d/2)**2 - (D_s/2)**2 + (D_d/2)**2 + 2*(D_d/2)*t_d),  
    S_ws  == 2*np.pi*(D_s/2)*t_s,                                       
    S_wf  == np.pi*((D_f/2)**2 - (D_s/2)**2 + 2*(D_f/2)*h_f),             
    S_w   == S_wd + S_ws + S_wf,
    P_move== rho_w*C_d*S_w*v**3/(2*eta_m)*1e-3
)

l_c, S, D_T, L_s, G_r, G_t, Pcomms = symbolic('l_c', 'S', 'D_T', 'L_s', 'G_r', 'G_t', 'P_{comms}')
db2dec = lambda x: 10 ** (x / 10)   
dec2db = lambda x: 10*log(abs(x), 10)
k, c, f, eta_parab, D_r, T_s, theta_t, error_t, L_pt_db, h, Re, L_a, L_l, L_p, R, E_N = 1.38065e-23, 3.0e8, 2.2e9, 0.55, 0.30, 135, 32, 27, -12 * (27 / 32) ** 2, 780e3, 6378e3, db2dec(-0.3), db2dec(-1.0), db2dec(-0.1), 10e6, 10
l_c = c / f
S = np.sqrt(h*(h+2*Re))
L_pt_db =  -12*(error_t/theta_t)**2
Comms = ConstraintSet(               
    G_t == eta_parab*(np.pi*d/l_c)**2,
    G_r == eta_parab*(np.pi*D_r/l_c)**2,
    L_s == (l_c / (4 * np.pi * S)) ** 2,
    Pcomms == E_N/(L_a*L_s*L_l*L_p*G_r*G_t)*(k*T_s*R),
)

t_mission, t_comms, t_move, t_service, t_recharge = 24, 1, 1, 12, 12  # [hr]
P_hotel, E_AUV, gamma = 50, 1.9, 1  # W, kWhr, –
P_move, P_service, E_required, E_move, E_hotel, E_comms, E_service = symbolic('P_{move}', 'P_{service}', 'E_{required}', 'E_{move}', 'E_{hotel}', 'E_{comms}', 'E_{service}')
Power = ConstraintSet(
    E_move     == P_move  * t_move,                       # P_move assumed in kW
    E_hotel    == P_hotel * t_mission * 1e-3,         # Whr to kWhr
    E_comms    == Pcomms  * t_comms   * 1e-3,         # Whr to kWhr
    E_service  == E_AUV   * gamma,                        # kWhr
    P_service  == E_service / t_service,                  # kW
    E_required == E_hotel + E_move + E_service + E_comms
)

E_recharge, P_recharge = symbolic('E_{recharge}', 'P_{recharge}')
eta_s, phi_s, theta_deg, Ideg, ddeg, Lsolar = 0.27, 800, 55, 0.9, 0.005, 10 
Solar = ConstraintSet(
    E_recharge == E_required,  # kW*h
    P_recharge == E_recharge / t_recharge,  # kW
    A_solar == P_recharge*1e3 / (eta_s * phi_s * np.cos(theta_deg * np.pi/180) * Ideg * (1 - ddeg)**Lsolar) # m^2
)

C,  = symbolic('C')
mu_batt, DOD, eta_batt, N, m_batt_zero = 200, 0.7, 0.85, 1, 5 
Battery = ConstraintSet(
    C == E_required / (DOD * N * eta_batt),  
    m_batt == C * 1e3 / mu_batt +m_batt_zero # kg
)
FPF = FunctionalSet(Geometry, Hydro, Mass, Propulsion, Comms, Power, Solar, Battery).subsetof(
    h_f <= 0.9*t_f, 
    D_s <= 0.9*D_f, 
    D_s <= 0.9*D_d, 
    0.1 <= P_move, 
    50 <= Pcomms).minimize(m_platform)
FPF_MDA = FPF.config(elim = [Geometry, FunctionalSet([Hydro, Mass]).config(parallel=[Hydro,Mass]), Propulsion, Comms, Power, Solar, Battery])

In [61]:
f_MDF = FPF_MDA.build()

[f_MDF.idxrev[elt] for elt in get_sharedvars([elt.build(indices=f_MDF.indices).analysis for elt in FPF.supersets])]

[F_W, D_f, P_{move}, P_{comms}, m_{batt}, E_{required}, A_{solar}, d, t_d]

In [62]:
f_MDF = FPF_MDA.build()
x0 = load_vals('applications/pearl/pearl_data/pearl_initial', 
          {str(key).replace('{','').replace('}',''):val for key,val in f_MDF.indices.items()}, path_to_file='')
x0_MDA = f_MDF.analysis(x0)
for elt in FPF.supersets:
    fP = elt.build()
    xP = load_vals(unload_vals(x0_MDA, f_MDF.indices), fP.indices, isdict=True)
    print_formatted_table([xP], fP.indices)

d     D_f   A_{solar}
0.058 1.158 3.149    
F_W   F_B   V_d   h_f  D_f   D_s t_s t_d   D_d 
77.05 77.05 7.862 0.45 1.158 2   2   0.901 1.25
t_d   m_{struct} t_f D_f   t_s D_s D_d  m_{batt} m_{platform} m_{solar} A_{solar} F_W  
0.901 7751.292   0.5 1.158 2   2   1.25 38.678   7862.292     10        3.149     77.05
P_{move} v   S_{w}  S_{wf} S_{wd} S_{ws} h_f  D_f   D_s t_s t_d   D_d 
9.98e-03 0.1 14.964 -0.452 2.85   12.566 0.45 1.158 2   2   0.901 1.25
P_{comms} G_t   G_r    L_s d    
897.688   0.978 26.273 0   0.058
E_{required} E_{hotel} E_{service} E_{move} E_{comms} P_{service} P_{comms} P_{move}
4.008        1.2       1.9         9.98e-03 0.898     0.158       897.688   9.98e-03
A_{solar} P_{recharge} E_{recharge} E_{required}
3.149     0.334        4.008        4.008       
m_{batt} C     E_{required}
38.678   6.736 4.008       
